# Prognozowanie miesięcznej liczby szkód komunikacyjnych za pomocą PROC FORECAST

## Podsumowanie wykonawcze

Zespół rezerw aktuarialnych potrzebuje 12-miesięcznej prognozy miesięcznej liczby szkód komunikacyjnych, uwzględniającej stały wzrost portfela oraz wyraźny wzrost zimowy. Ten notatnik generuje pięć lat syntetycznych miesięcznych liczb szkód (60 miesięcy, od stycznia 2020 do grudnia 2024, w zakresie od około 1460 do 2780 szkód), a następnie używa **PROC FORECAST** do dopasowania bazowego modelu stepwise-autoregresyjnego oraz multiplikatywnego modelu sezonowego Holta-Wintersa. Każdy model generuje 12 miesięcznych prognoz punktowych z 95% przedziałami ufności do planowania zdolności i rezerw. Sezonowy model Holta-Wintersa przewiduje najsilniejszy popyt jeden do dwóch miesięcy naprzód (około 2780-2900 szkód), łagodnie opadając do minimum około kroku dziewiątego (około 2200), podczas gdy model autoregresyjny przewiduje łagodniejszy spadek; oba przedziały ufności rozszerzają się wraz z horyzontem prognozy.

## Źródła danych

| Zbiór danych | Wiersze | Ziarnistość | Kluczowe zmienne | Opis |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | Jeden wiersz na miesiąc kalendarzowy (sty 2020 - gru 2024) | `date` (data SAS, `MONYY7.`), `claim_count` (liczbowa) | Syntetyczna miesięczna liczba szkód komunikacyjnych zbudowana z liniowego trendu wzrostu (~9 szkód/miesiąc), sinusoidalnego cyklu rocznego, addytywnego wzrostu zimowego (gru/sty/lut) oraz szumu gaussowskiego (`rand('normal')`). Ziarno `20240531` zapewnia powtarzalność. |

# Prognozowanie miesięcznej liczby szkód komunikacyjnych

Zespoły rezerw i planowania zdolności w towarzystwie ubezpieczeń osobistych potrzebują prognozy liczby **szkód komunikacyjnych** zgłaszanych każdego miesiąca. Wolumen szkód w tym portfelu rośnie stale wraz z ekspansją portfela i gwałtownie wzrasta każdej zimy, gdy lód, śnieg i krótszy dzień zwiększają liczbę szkód kolizyjnych i szybowych.

Ten notatnik krok po kroku przedstawia pełny przepływ pracy `PROC FORECAST`:

1. Wygeneruj realistyczny, w pełni syntetyczny szereg miesięcznej liczby szkód.
2. Dopasuj bazowy model **stepwise-autoregresyjny (STEPAR)**, który uwzględnia trend i autokorelację.
3. Dopasuj **multiplikatywny model Holta-Wintersa (WINTERS)**, który jawnie modeluje 12-miesięczny cykl sezonowy.
4. Porównaj oba modele i zinterpretuj prognozę oraz przedział ufności.

Wszystko działa na wbudowanych danych syntetycznych — bez plików zewnętrznych ani dostępu do sieci.

## Krok 1 — Generowanie syntetycznego szeregu szkód

Poniższy krok DATA buduje **60 obserwacji miesięcznych** (od stycznia 2020 do grudnia 2024). Dla każdego miesiąca łączymy cztery składowe odzwierciedlające prawdziwy portfel komunikacyjny:

- **Trend** — baza ~1800 szkód, rosnąca o ~9 miesięcznie wraz ze wzrostem ekspozycji.
- **Cykl roczny** — składnik sinus/cosinus dający gładką falę sezonową.
- **Wzrost zimowy** — addytywny skok w grudniu, styczniu i lutym.
- **Szum** — `rand('normal', 0, 90)` dla realistycznej zmienności miesiąc do miesiąca.

`call streaminit()` ustala strumień losowy, dzięki czemu notatnik jest powtarzalny. Zmienna `date` to prawdziwa data SAS zbudowana za pomocą `INTNX` i sformatowana `MONYY7.`, a instrukcja `ID date INTERVAL=MONTH` wskazuje ją jako identyfikator czasu. Pierwsze 14 wierszy wydrukowanych poniżej mieści się w przedziale około 1460-2450 szkód.

In [1]:
DANE claims;
    CALL streaminit(20240531);
    POWTÓRZ i = 0 TO 59;
        /* Zbuduj prawdziwą datę SAS, aby INTERVAL=MONTH się zgadzało */
        date = intnx('month', '01JAN2020'd, i);
        format date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = sty ... 12 = gru */

        trend  = 1800 + 9 * i;               /* rosnąca baza ekspozycji */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx IN (12, 1, 2));   /* wzrost od lodu/śniegu */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        JEŚLI claim_count < 0 WTEDY claim_count = 0;
        WYJŚCIE;
    KONIEC;
    ZACHOWAJ date claim_count;
WYKONAJ;

PROCEDURA DRUKUJ DANE=claims(obs=14) ETYKIETA;
    ETYKIETA date = 'Miesiąc' claim_count = 'Zgłoszone szkody';
    TYTUŁ 'Pierwsze 14 miesięcy syntetycznych zgłoszeń szkód komunikacyjnych';
WYKONAJ;

                           Pierwsze 14 miesięcy syntetycznych zgłoszeń szkód komunikacyjnych                            

  Obs   Miesiąc   Zgłoszone szkody
    1     21915               2178
    2     21946               2281
    3     21975               2252
    4     22006               1974
    5     22036               2067
    6     22067               1805
    7     22097               1697
    8     22128               1619
    9     22159               1462
   10     22189               1688
   11     22220               1713
   12     22250               2008
   13     22281               2116
   14     22312               2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Krok 2 — Bazowy model stepwise-autoregresyjny (METHOD=STEPAR)

`METHOD=STEPAR` jest metodą domyślną. Najpierw dopasowuje model trendu czasowego — tutaj `TREND=2` oznacza trend liniowy — a następnie stosuje **stepwise autoregresję** do reszt, wprowadzając i zachowując opóźnienia AR na podstawie istotności. `AR=4` ogranicza kandydujący rząd autoregresji do czterech opóźnień, co w zupełności wystarcza dla szeregu miesięcznego z krótkoterminowym momentum.

Kluczowe opcje użyte tutaj:

- `LEAD=12` — prognoza 12 miesięcy poza danymi.
- `ALPHA=0.05` — 95% przedziały ufności.
- `OUTFULL` — łączy historyczne wiersze `ACTUAL` z wierszami horyzontu prognozy w zbiorze `OUT=` (zamiast samych wierszy prognozy).
- `OUTLIMIT` — dodaje **kolumny** dolnej/górnej granicy ufności (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` — nazywa identyfikator czasu i deklaruje, że szereg jest miesięczny.

In [2]:
PROCEDURA forecast DANE=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    ZMIENNA claim_count;
WYKONAJ;

PROCEDURA DRUKUJ DANE=fc_stepar(obs=10) ETYKIETA;
    TYTUŁ 'Wynik STEPAR: wiersze rzeczywiste, dopasowane i prognozowane';
WYKONAJ;

                           Pierwsze 14 miesięcy syntetycznych zgłoszeń szkód komunikacyjnych                            

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                              Wynik STEPAR: wiersze rzeczywiste, dopasowane i prognozowane                              

  Obs   DATE  _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT
    1  21915  ACTUAL  2121.816667                .                .
    2  21946  ACTUAL  2167.711419                .                .
    3  21975  ACTUAL  2254.781177                .                .
    4  22006  ACTUAL  2228.505912                .                .
    5  22036  ACTUAL  1978.152296                .                .
    6  22067  ACTUAL  2030.606563                .                .
    7  22097  ACTUAL  1864.520418                .                .
    8  22128  ACTUAL  1784.855682                .                .
    9  22159  ACTUAL  1740.781553  


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### Odczyt zbioru OUT=

Zbiór `OUT=` układa wiersze według kolumny `_TYPE_` i przenosi przedziały ufności jako dodatkowe **kolumny**:

| Element | Rodzaj | Znaczenie |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | wiersz | Zaobserwowana wartość `claim_count` dla każdego z 60 historycznych miesięcy. |
| `_TYPE_ = 'FORECAST'` | wiersz | 12 prognoz punktowych dla horyzontu prognozy. |
| `L95_claim_count` / `U95_claim_count` | kolumna | Dolna / górna granica 95% przedziału ufności, wypełniona w wierszach `FORECAST` (brak w wierszach `ACTUAL`). Poziom liczbowy odzwierciedla `ALPHA=`. |

Wydrukowany tutaj zbiór `OUT=` zawiera więc 72 wiersze: 60 historycznych wierszy `ACTUAL`, po których następuje 12 wierszy horyzontu `FORECAST`. Pierwsze dziesięć wierszy pokazanych powyżej to same wiersze `ACTUAL`, z brakującymi kolumnami przedziału ufności, ponieważ granice dotyczą tylko wierszy prognozy.

> **Uwaga dotycząca silnika.** SAS `OUTFULL` zapisuje również wewnątrzpróbkowy jednokrokowy wiersz `FORECAST` dla każdego historycznego miesiąca, a `OUTRESID` dodaje wiersze `_TYPE_='RESIDUAL'`. PROC FORECAST w Jennerze nie generuje jeszcze tych dopasowanych wierszy/reszt wewnątrzpróbkowych (śledzone jako test luki `400979`), więc ten notatnik odczytuje jedynie historię `ACTUAL` oraz prognozę `FORECAST` w przód.

## Krok 3 — Sezonowy model Holta-Wintersa (METHOD=WINTERS)

Model STEPAR uchwytuje trend i krótkoterminową autokorelację, ale nie modeluje jawnie powtarzającego się wzrostu grudzień-luty. Dla szeregu z wyraźnym rocznym rytmem lepszym narzędziem jest **multiplikatywny model Holta-Wintersa**.

`METHOD=WINTERS` dekomponuje szereg na poziom, trend liniowy (`TREND=2`) oraz **multiplikatywny czynnik sezonowy**. `SEASONS=12` deklaruje 12-okresowy (miesięczny) cykl sezonowy. Ponownie prosimy o 12-miesięczny `LEAD` z 95% granicami oraz `OUTFULL OUTLIMIT`, aby wynik odpowiadał uruchomieniu STEPAR.

Ponieważ silnik przesuwa prognozowany `ID` o jedną jednostkę na krok (zamiast honorować `INTERVAL=MONTH` dla dat horyzontu — test luki `400979`), poniższa komórka przegląda prognozę według **miesięcy naprzód (krok 1-12)** zamiast polegać na wydrukowanych datach kalendarzowych.

In [3]:
PROCEDURA forecast DANE=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    ZMIENNA claim_count;
WYKONAJ;

/* Zachowaj 12-miesięczny horyzont prognozy i indeksuj go krokiem (1..12). */
DANE horizon;
    USTAW fc_winters;
    PRZECHOWAJ months_ahead 0;
    JEŚLI _type_ = 'FORECAST';
    months_ahead + 1;
    ZACHOWAJ months_ahead claim_count l95_claim_count u95_claim_count;
WYKONAJ;

PROCEDURA DRUKUJ DANE=horizon ETYKIETA;
    ETYKIETA months_ahead     = 'Miesięcy naprzód'
          claim_count       = 'Prognozowane szkody'
          l95_claim_count   = 'Dolna granica 95%'
          u95_claim_count   = 'Górna granica 95%';
    TYTUŁ 'Prognoza Holta-Wintersa na 12 miesięcy naprzód (wg kroku)';
WYKONAJ;

                              Wynik STEPAR: wiersze rzeczywiste, dopasowane i prognozowane                              

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                               Prognoza Holta-Wintersa na 12 miesięcy naprzód (wg kroku)                                

  Obs    Miesięcy naprzód  Prognozowane szkody  Dolna granica 95%   Górna granica 95%
    1                   1           2783.07951        2577.844742         2988.314278
    2                   2          2897.355557        2607.109764         3187.601349
    3                   3          2805.272075        2449.795029          3160.74912
    4                   4          2664.498049        2254.028514         3074.967585
    5                   5          2628.810136        2169.891244         3087.729029
    6                   6           2548.39319        2045.672732         3051.113649
    7                   7          2391.64975


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Krok 4 — Bezpośrednie porównanie obu modeli

Najprostszym sposobem oceny, czy model sezonowy się opłaca, jest zestawienie jego 12-miesięcznej prognozy z bazowym modelem STEPAR, krok po kroku. Pobieramy wiersze `FORECAST` z obu zbiorów `OUT=`, indeksujemy każdy według miesięcy naprzód i łączymy je, aby rozbieżność była widoczna na pierwszy rzut oka.

Obie metody zgadzają się co do ogólnego poziomu, ale różnią się **kształtem**: Holt-Winters przenosi roczny rytm w horyzont prognozy (wyższy poziom na początku horyzontu, który łagodnieje do minimum w połowie horyzontu i ponownie rośnie), podczas gdy STEPAR — który modeluje sezonowość jedynie pośrednio poprzez opóźnienia AR — opada łagodniej w kierunku średniej szeregu. Różnica między nimi na każdym kroku to sygnał sezonowy, którego STEPAR nie uwzględnia.

> SAS zwykle napędzałby tę kontrolę adekwatności jednokrokowymi resztami `OUTRESID` (`_TYPE_='RESIDUAL'`). Jenner nie wypełnia jeszcze tych wierszy (test luki `400979`), więc zamiast tego porównujemy bezpośrednio obie prognozy w przód — diagnostykę wykorzystującą wyłącznie dane, które silnik faktycznie generuje.

In [4]:
/* Horyzont STEPAR, indeksowany miesiącami naprzód. */
DANE stepar_h;
    USTAW fc_stepar;
    PRZECHOWAJ months_ahead 0;
    JEŚLI _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    ZACHOWAJ months_ahead stepar;
WYKONAJ;

/* Horyzont WINTERS, indeksowany miesiącami naprzód. */
DANE winters_h;
    USTAW fc_winters;
    PRZECHOWAJ months_ahead 0;
    JEŚLI _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    ZACHOWAJ months_ahead winters;
WYKONAJ;

/* Połącz oba i pokaż lukę sezonową pomijaną przez STEPAR. */
DANE compare;
    POŁĄCZ stepar_h winters_h;
    WEDŁUG months_ahead;
    seasonal_gap = winters - stepar;
WYKONAJ;

PROCEDURA DRUKUJ DANE=compare ETYKIETA;
    ETYKIETA months_ahead = 'Miesięcy naprzód'
          stepar        = 'Prognoza STEPAR'
          winters       = 'Prognoza Wintersa'
          seasonal_gap  = 'Winters minus STEPAR';
    format stepar winters seasonal_gap 8.0;
    TYTUŁ 'STEPAR a Holt-Winters: porównanie prognoz na 12 miesięcy';
WYKONAJ;

                                STEPAR a Holt-Winters: porównanie prognoz na 12 miesięcy                                

  Obs    Miesięcy naprzód  Prognoza STEPAR  Prognoza Wintersa  Winters minus STEPAR
    1                   1             2619               2783                   164
    2                   2             2537               2897                   361
    3                   3             2384               2805                   421
    4                   4             2214               2664                   450
    5                   5             2089               2629                   540
    6                   6             2010               2548                   539
    7                   7             1982               2392                   410
    8                   8             1995               2240                   245
    9                   9             2031               2197                   166
   10                  10             


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Krok 5 — Prognoza dla każdej linii biznesowej naraz (przetwarzanie BY)

Prawdziwe uruchomienia rezerw obejmują kilka produktów. Gdy dane są posortowane według `line_of_business`, instrukcja `BY` sprawia, że `PROC FORECAST` dopasowuje **niezależny model sezonowy dla każdej grupy** w jednym wywołaniu. Tutaj syntetyzujemy portfel Auto (wyższy wolumen bazowy) i portfel Dom (niższy bazowy) i prognozujemy oba na sześć miesięcy naprzód. `OUTALL` zapisuje pełny zestaw wierszy — historię `ACTUAL` i horyzont `FORECAST` — wraz z kolumnami granic dla każdej grupy, a my zachowujemy jedynie wiersze `FORECAST` dla poniższej tabeli.

In [5]:
DANE multi_book;
    CALL streaminit(20240531);
    DŁUGOŚĆ line_of_business $6;
    POWTÓRZ lob = 1 TO 2;
        JEŚLI lob = 1 WTEDY line_of_business = 'Auto';
        PRZECIWNIE            line_of_business = 'Dom';
        POWTÓRZ i = 0 TO 47;
            date = intnx('month', '01JAN2021'd, i);
            format date monyy7.;
            mi   = mod(i, 12) + 1;
            base = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(base + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi IN (12, 1, 2))
                + rand('normal', 0, 70));
            WYJŚCIE;
        KONIEC;
    KONIEC;
    ZACHOWAJ line_of_business date claim_count;
WYKONAJ;

PROCEDURA SORTUJ DANE=multi_book;
    WEDŁUG line_of_business date;
WYKONAJ;

PROCEDURA forecast DANE=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    WEDŁUG line_of_business;
    id date interval=month;
    ZMIENNA claim_count;
WYKONAJ;

PROCEDURA DRUKUJ DANE=fc_by(obs=12) ETYKIETA;
    ETYKIETA line_of_business = 'Linia biznesowa'
          claim_count       = 'Liczba szkód';
    GDZIE _type_ = 'FORECAST';
    TYTUŁ 'Prognozy 6-miesięczne wg linii biznesowej';
WYKONAJ;

                                STEPAR a Holt-Winters: porównanie prognoz na 12 miesięcy                                


BY Group: line_of_business=Auto

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Dom

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                       Prognozy 6-miesięczne wg linii biznesowej                                        

  Obs  Linia biznesowa   DATE    _TYPE_   Liczba szkód  L95_CLAIM_COUNT  U95_CLAIM_COUNT  RESIDUAL_CLAIM_COUNT
    1  Auto             23742  FORECAST    2524.596717      2359.095846      2690.097589                     .
    2  Auto             23773  FORECAST    2604.418724      2370.365147        2838.4723                     .
    3  Auto             23801  FORECAST    2576.092313      2289.436395       2862.74823                     .
    4  Auto             23832  


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=Auto
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=Dom
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## Interpretacja wyników

**Co modele mówią zespołowi rezerw:**

- **STEPAR** odtwarza rosnący trend i krótkoterminowe momentum, ale jego 12-miesięczna prognoza łagodnie opada z około 2620 (krok 1) do około 1980 w połowie horyzontu, po czym wraca do około 2140 — nie odtwarza ostrego szczytu zimowego, ponieważ traktuje sezonowość jedynie poprzez opóźnienia autoregresyjne. Jest to rozsądny szybki model bazowy.
- **WINTERS** z `SEASONS=12` przenosi roczny rytm bezpośrednio poprzez swoje multiplikatywne czynniki sezonowe: jego prognoza jest najwyższa na początku horyzontu (około 2780 w kroku 1, około 2900 w kroku 2), łagodnieje do minimum około kroku 9 (około 2200) i ponownie rośnie do kroku 12 (około 2800). Względem bazowego modelu STEPAR jest **o 150-660 szkód wyższa** przez większość horyzontu (patrz kolumna `Winters - STEPAR` w Kroku 4) — ta różnica to sygnał sezonowy pomijany przez model autoregresyjny.
- **Przedział ufności 95%** (kolumny `L95_*`/`U95_*`, kontrolowane przez `ALPHA=`) rozszerza się wraz z horyzontem — dla modelu WINTERS od szerokości około 410 szkód w kroku 1 do około 1420 w kroku 12 — uczciwy sygnał, że szacunki na dalszym horyzoncie niosą większą niepewność niż te bliskoterminowe. Zespół rezerw powinien utrzymywać kapitał względem górnej granicy, a nie tylko prognozy punktowej.
- **Przetwarzanie BY** generuje prognozy Auto i Dom w jednym przebiegu, każdą z własnym dopasowaniem sezonowym. Portfel Auto prognozuje w zakresie około 2510-2600, podczas gdy portfel Dom mieści się w okolicach 1870-2030, więc każda linia zachowuje swój własny poziom i kształt sezonowy — wzorzec, który zespół rozszerzyłby na każdy produkt w portfelu.

**Wniosek:** dla szeregu liczby szkód z wyraźnym cyklem rocznym, `METHOD=WINTERS SEASONS=12` jest naturalnym wyborem; model bazowy STEPAR stanowi użyteczną kontrolę poprawności, a `OUTFULL`/`OUTLIMIT` wraz z porównaniem modeli krok po kroku pozwalają aktuariuszowi oszacować rezerwę wraz z jej przedziałem niepewności.

> **Uwaga o wierności silnika.** Ten notatnik dokumentuje dwa obecne ograniczenia PROC FORECAST w Jennerze (test luki `400979`): prognozowany `ID` jest przesuwany o jedną jednostkę na krok zamiast o `INTERVAL=MONTH`, więc wydrukowane daty prognozy nie są zamierzonymi miesiącami kalendarzowymi 2025 (zamiast tego przeglądamy horyzont według kroku); oraz `OUTRESID`/`OUTALL` nie wypełniają jeszcze jednokrokowych wierszy `_TYPE_='RESIDUAL'`, więc diagnostyka reszt jest zastąpiona bezpośrednim porównaniem dwóch modeli. Same prognozy punktowe i przedziały ufności są generowane i to na nich opiera się powyższa narracja.